In [1]:
import pandas as pd
from google.colab import auth
import gspread
from google.auth import default

# Authenticate the user
auth.authenticate_user()

# Authorize gspread
creds, _ = default()
gc = gspread.authorize(creds)

# Open the Google Sheet by URL
sheet_url = 'https://docs.google.com/spreadsheets/d/1X5Iz3yL7Oq_l5a9hylxNYnA7qHzctnIxexNSRzwEdRE/edit?usp=drive_link'
sh = gc.open_by_url(sheet_url)

# Get the first worksheet
worksheet = sh.get_worksheet(0)

# Get all values from the worksheet
rows = worksheet.get_all_values()

# Convert to DataFrame (assuming the first row contains headers)
df = pd.DataFrame(rows[1:], columns=rows[0])

# Display the first few rows and info
print(df.head())
print(df.info())

                                                text label_specific  \
0  Hi I am running the IR test program from Max D...    human_legit   
1  Security Note: Trade Me will never ask you for...    human_legit   
2  Trade Me Offer RequestGenerated 8 December, 6:...    human_legit   
3  Hi Tony Not sure why it didn't work, but I man...    human_legit   
4  Kindly suggest changes -----------------------...    human_legit   

  label_generic  
0         legit  
1         legit  
2         legit  
3         legit  
4         legit  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   text            4000 non-null   object
 1   label_specific  4000 non-null   object
 2   label_generic   4000 non-null   object
dtypes: object(3)
memory usage: 93.9+ KB
None


In [2]:
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label_specific'],
    random_state=42
)

# Display the shapes of the resulting sets
print("Training set shape:", train_df.shape)
print("Testing set shape:", test_df.shape)

# Verify the class distribution in the training set
print("\nTraining set class distribution (normalized):")
print(train_df['label_specific'].value_counts(normalize=True))

# Verify the class distribution in the testing set
print("\nTesting set class distribution (normalized):")
print(test_df['label_specific'].value_counts(normalize=True))

Training set shape: (3200, 3)
Testing set shape: (800, 3)

Training set class distribution (normalized):
label_specific
llm_phishing      0.25
llm_legit         0.25
human_legit       0.25
human_phishing    0.25
Name: proportion, dtype: float64

Testing set class distribution (normalized):
label_specific
human_legit       0.25
human_phishing    0.25
llm_phishing      0.25
llm_legit         0.25
Name: proportion, dtype: float64


In [3]:
train_df.to_csv('DBTL2_preprocessed_train_valid_set80.csv', index=False) #Train set
test_df.to_csv('DBTL2_preprocessed_evaluation_set20_50.csv', index=False) #50% legit, 50% phising eval set

In [6]:
def create_imbalanced_test_set(df, target_percent, label_col='label_generic', stratify_col='label_specific'):
    """
    Filters df so that target_percent of the result is 'phishing',
    while maintaining 'label_specific' proportions for all rows.
    """
    # 1. Separate phishing from non-phishing
    phish_df = df[df[label_col] == 'phishing']
    clean_df = df[df[label_col] != 'phishing']

    # 2. Calculate how many phishing samples we need
    # If P is target percent, then: Phish_Count / (Phish_Count + Clean_Count) = P
    # Solving for Phish_Count: Phish_Count = (P * Clean_Count) / (1 - P)
    n_clean = len(clean_df)
    n_phish_needed = int((target_percent * n_clean) / (1 - target_percent))

    # 3. Sample the phishing rows while stratifying by the specific labels
    # We use train_test_split as a shortcut to sample with stratification
    _, phish_sampled = train_test_split(
        phish_df,
        test_size=n_phish_needed,
        stratify=phish_df[stratify_col],
        random_state=42
    )

    # 4. Combine and shuffle
    final_df = pd.concat([phish_sampled, clean_df]).sample(frac=1, random_state=42)
    return final_df

# Generate the 5% dataset
test_df_5pct = create_imbalanced_test_set(test_df, 0.05)

# Generate the 1% dataset
test_df_1pct = create_imbalanced_test_set(test_df, 0.01)

In [7]:
test_df_5pct.to_csv('DBTL2_preprocessed_evaluation_set20_5.csv', index=False) #95% legit, 5% phising eval set
test_df_1pct.to_csv('DBTL2_preprocessed_evaluation_set20_1.csv', index=False) #99% legit, 1% phising eval set